In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z tabulky
def nacteni_array(df, *list_data, dtype=float):
    return tuple(df[v].dropna().to_numpy(dtype=dtype) for v in list_data)

# nacteni z df do numpy
def df_to_numpy(df, sloupce, od=0, do=None, dtype=float):
    """
    sloupce: jmeno sloupce (str) nebo seznam jmen (list[str])
    od, do:  rozsah radku podle pozice (od vcetne, do bez); do=None znamená až do konce
    """
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    data = df[sloupce].iloc[od:do].to_numpy(dtype=dtype)
    return data[:, 0] if len(sloupce) == 1 else data

# ufloat pro tuple
def tuple_unc_A(*velicina):
    list_velicin = [*velicina]
    return tuple(uf(np.mean(v), unc_A(v)) for v in list_velicin)

In [ ]:
# nacteni google tabuleksach15_proc
sheet_id = '1SR-xU3o3py2D8sSCnXWfbbmhoSYlBci-QvDomAE58Fw'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_refrakto = pd.read_csv(url)
df_refrakto.dropna()

In [ ]:
# refrakto a polari
destil_n_tabulky = uf(1.3330, 0.0003)
destil_n, sach5_n, sach10_n, sach15_n = (df_refrakto[c].dropna().to_numpy() for c in ['destil_n', 'sach5_n', 'sach10_n', 'sach15_n'])
destil_n, sach5_n, sach10_n, sach15_n = (uf(np.mean(v), unc_A(v)) for v in [destil_n, sach5_n, sach10_n, sach15_n])

destil_proc, sach5_proc, sach10_proc, sach15_proc = (df_refrakto[c].dropna().to_numpy() for c in ['destil_proc', 'sach5_proc', 'sach10_proc', 'sach15_proc'])
destil_proc, sach5_proc, sach10_proc, sach15_proc = (uf(np.mean(v), unc_A(v)) for v in [destil_proc, sach5_proc, sach10_proc, sach15_proc])

posun_n = destil_n_tabulky - destil_n
destil_n, sach5_n, sach10_n, sach15_n = (v + posun_n for v in [destil_n, sach5_n, sach10_n, sach15_n])

def sach_N(n, a=uf(0.00140, 0.00003), b=destil_n_tabulky):
    c = (n-b)/a
    return c

sach5_proc, sach10_proc, sach15_proc = (sach_N(v) for v in [sach5_n, sach10_n, sach15_n])
sach5_proc, sach10_proc, sach15_proc

In [ ]:
# nacteni google tabuleksach15_proc
sheet_id = '1SR-xU3o3py2D8sSCnXWfbbmhoSYlBci-QvDomAE58Fw'
gid = '415661874'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_polari = pd.read_csv(url)
df_polari.dropna()

In [ ]:

spec_stac_tabulkova = 66.53
nazvy = ['Nulová poloha (°)', 'Roztok 15 % – odečtený úhel (°)', 'Roztok 10 % – odečtený úhel (°)', 'Roztok 5 % – odečtený úhel (°)']
nul_poloha, uhel15, uhel10, uhel5 = df_to_numpy(df_polari, nazvy).T
nul_poloha, uhel15, uhel10, uhel5 = tuple_unc_A(nul_poloha, uhel15, uhel10, uhel5)

d = 1 # v dm
konc = np.array([sach5_proc, sach10_proc, sach15_proc])
alpha = np.array([uhel5, uhel10, uhel15])

spec_stac = (100*alpha)/(d*konc)
print(spec_stac)

In [ ]:
# nacteni google tabuleksach15_proc
sheet_id = '1SR-xU3o3py2D8sSCnXWfbbmhoSYlBci-QvDomAE58Fw'
gid = '2139253552'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_malus = pd.read_csv(url)
#df_malus.dropna()

In [ ]:
names_malus = ['Úhel otočení polarizátoru (°)', 'bílé světlo muA', 'filtr 10 nA']
uhel_pol, bile, zelene = df_to_numpy(df_malus, names_malus, od=1).T
uhel_pol = uhel_pol - 18
uhel_pol_ls = np.linspace(-8, 352, 1000)
zelene = zelene * 10

def cos2(phi, Imax, Imin):
    return Imin + ((Imax - Imin)*np.cos(phi*np.pi/180)**2)

p0_bile = [700,600]
popt_bile, pcov_bile = curve_fit(cos2, uhel_pol, bile, p0=p0_bile)
Imax_bile, Imin_bile = popt_bile
unc_Imax_bile, unc_Imin_bile = np.sqrt(np.diag(pcov_bile))

p0_zelene = [600,30]
popt_zelene, pcov_zelene = curve_fit(cos2, uhel_pol, zelene, p0=p0_zelene)
Imax_zelene, Imin_zelene = popt_zelene
unc_Imax_zelene, unc_Imin_zelene = np.sqrt(np.diag(pcov_zelene))
Imax_zelene_unc = uf(Imax_zelene, unc_Imax_zelene)
Imin_zelene_unc = uf(Imin_zelene, unc_Imin_zelene)
V = (Imax_zelene_unc - Imin_zelene_unc)/(Imax_zelene_unc + Imin_zelene_unc)
print(f'${V:.1uL}$')

fig, ax = plt.subplots(2 ,1 ,figsize=(16, 9), sharex=True)
ax[0].set_ylabel(r'$I$ - bílé světlo [$\mu$A]', fontsize=18)
ax[1].set_ylabel(r'$I$ - monochromatické světlo [nA]', fontsize=18)
ax[1].set_xlabel(r'$\varphi$', fontsize=18)
ax[0].scatter(uhel_pol, bile, label='Naměřené hodnoty', s=50)
ax[1].scatter(uhel_pol, zelene, label='Naměřené hodnoty', s=50, color='green')
ax[0].plot(uhel_pol_ls, cos2(uhel_pol_ls, Imax_bile, Imin_bile), label='Fit')
ax[1].plot(uhel_pol_ls, cos2(uhel_pol_ls, Imax_zelene, Imin_zelene), label='Fit', color='green')
ax[0].tick_params(labelsize=12)
ax[1].tick_params(labelsize=12)
ax[0].grid(True, alpha=0.7)
ax[1].grid(True, alpha=0.7)
ax[0].legend(fontsize=12)
ax[1].legend(fontsize=12)
plt.savefig(downloads_dir / 'malus.pdf', dpi=300, bbox_inches='tight')